In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/05_model_input/6_final_NGFS_viable_scenarios.csv")
df = df.loc[df["sector"] == "Power",:]
df = df.loc[df["technology"].isin([ 'CoalCap', 'GasCap', 'CoalCap - w/o CCS', 'GasCap - w/o CCS' ]),:]

# Compatible scenarios

In [3]:
# the EBITDA is likely to be negative when the values in fom_usd_per_mw_yr are higher
#  than capacity_factor * hours_per_year * scenario_price
incompatible_fixed_cost = df["om_cost_usd_per_mw_per_yr"] > df["scenario_capacity_factor"] * (24*365) * df["scenario_price"]

# the EBITDA is likely to be negative when fuel_price/efficiency > scenario_price .
# incompatible_var_cost = df["fuel_price"] / df["efficiency_decimal"] > (df["scenario_price"]*2)
incompatible_var_cost = df["fuel_price"] * df["fuel_intensity"] > df["scenario_price"]


In [4]:
df = df.assign(
    incompatible_var_cost = incompatible_var_cost.astype(bool),
    incompatible_fixed_cost = incompatible_fixed_cost.astype(bool),
)



In [5]:
df.loc[df["scenario_type"] == "baseline", ["scenario_provider", "scenario"]].drop_duplicates()

,scenario_provider,scenario
211,GCAM 6.0 NGFS,Current Policies
1681,MESSAGEix-GLOBIOM 2.0-M-R12-NGFS,Current Policies
3139,REMIND-MAgPIE 3.3-4.8,Current Policies


In [6]:
pd.options.display.max_rows = 1000

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type","sector", "technology",  "scenario_geography",
    "incompatible_var_cost","incompatible_fixed_cost"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "any"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "any"),
)

incompatibility_flagged = incompatibility_flagged.assign(
    likely_incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] | incompatibility_flagged["incompatible_fixed_cost"]),
    incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] & incompatibility_flagged["incompatible_fixed_cost"]),
)


incompatibility_flagged.query("(incompatible_var_cost == False)")

n_techs  \
scenario_provider                                  scenario                                              
GCAM 6.0 NGFS                                      Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   
MESSAGEix-GLOBIOM 2.0-M-R12-NGFS                   Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   
REMIND-MAgPIE 3.3-4.8                              Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   
REMIND-MAgPIE 3.3-4.8 IntegratedPhysicalDamages... Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   

                                                                                               n_regions  \
scenario_provider                                  scenario                                                
GCAM 6.0 NGFS                                      Below 2°C                                           5   
                                                   Current Policies                                    5   
                                                   Delayed transition                                  5   
                                                   Fragmented World                                    5   
                                                   Low demand                                          5   
                                                   Nationally Determined Contributions (NDCs)          5   
                                                  

In [7]:
pd.options.display.max_rows = 1500

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type","sector", "technology",   "scenario_geography",
    "incompatible_var_cost","incompatible_fixed_cost"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "sum"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "sum"),
).assign(
    n_tech_regions = lambda x: x["n_techs"] * x["n_regions"],
    perc_incompatible_fixed_cost = lambda x: x["incompatible_fixed_cost"] / x["n_tech_regions"],
    perc_incompatible_var_cost = lambda x: x["incompatible_var_cost"] / x["n_tech_regions"]
)
print(incompatibility_flagged.shape)
# incompatibility_flagged.query("incompatible_fixed_cost < n_tech_regions/2")
# incompatibility_flagged.query("perc_incompatible_fixed_cost <= 0.25 & perc_incompatible_var_cost <= 0.25")
incompatibility_flagged.query("perc_incompatible_var_cost <= 0.25")

(28, 7)


n_techs  \
scenario_provider                                  scenario                                              
GCAM 6.0 NGFS                                      Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   
MESSAGEix-GLOBIOM 2.0-M-R12-NGFS                   Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   
REMIND-MAgPIE 3.3-4.8                              Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   
REMIND-MAgPIE 3.3-4.8 IntegratedPhysicalDamages... Below 2°C                                         2   
                                                   Current Policies                                  2   
                                                   Delayed transition                                2   
                                                   Fragmented World                                  2   
                                                   Low demand                                        2   
                                                   Nationally Determined Contributions (NDCs)        2   
                                                   Net Zero 2050                                     2   

                                                                                               n_regions  \
scenario_provider                                  scenario                                                
GCAM 6.0 NGFS                                      Below 2°C                                           5   
                                                   Current Policies                                    5   
                                                   Delayed transition                                  5   
                                                   Fragmented World                                    5   
                                                   Low demand                                          5   
                                                   Nationally Determined Contributions (NDCs)          5   
                                                  